# 02 · How many dimensions? sEMG synergies and eigenpostures

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 1 · Existence: is the EMG low-dimensional beyond chance?**

**Read first:** [Lee & Seung 1999](https://www.nature.com/articles/44565) (NMF), [Santello et al. 1998](https://www.jneurosci.org/content/18/23/10105) (eigenpostures), Cunningham & Yu 2014, *Dimensionality reduction for large-scale neural recordings* (Nat Neurosci). **Owner:** Person B

**We expect:** 3 to 5 NMF synergies explain the held-out EMG (VAF ≥ 90%), clearly better than random synergies of the same size, and k stays the same when whole stages are held out. **Wrong if:** we need close to 16 synergies, or random non-negative synergies do almost as well.

**Model:** `EMG ≈ C × W`: `W` (k × 16) holds the k synergies, `C` (T × k) says how active each one is at every
frame. You write **PCA and NMF yourself, in NumPy** (no scikit-learn for the core algorithms).

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref02.py`](ref02.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ✅ **Aim 1** (weeks 1–5): define stable, low-dimensional representations of sEMG *and* hand pose.

**Goal:** choose the two bottleneck sizes: **k** synergies for the sEMG input and **K_POSE** eigenpostures for the pose output, each with a fair reference point.

**Why:** H1 says squeezing both sides of the decoder through small representations cuts calibration on new users. Every later stage uses the k chosen here, so it must not depend on a generous metric or on the split.

**Milestones**
1. VAF vs k (k = 1 to 10) for NMF and PCA on test data, uncentred **and** centred
2. The same curve for random non-negative synergies (the null)
3. k chosen by leave-one-stage-out, with a 95% CI over held-out stage groups
4. A figure of the chosen synergies, and pose error (°) vs number of eigenpostures

**Gate (decides the next stage):** NMF beats random synergies by a clear margin (CI of the gap above 0) at the k chosen by leave-one-stage-out. If not, stop calling them synergies (stage 2 and 3 questions change).

**Done when:** a chosen k and K_POSE, each with its curve, its null and its CI.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref02 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (builds it with prep.py the first time) ─────────────
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):   # missing, or built by an older version of prep.py
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording
filename_of_rec = d["filename"]                   # (30,) file name: pairs the two simultaneous recordings

train, test = split == 0, split == 1
SEED = 0   # pass seed=SEED to every random start so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

### The variables you start with
`N` = number of frames in all 30 recordings together (105,484: about 35 minutes at 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float64 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float64 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |
| `filename_of_rec` | (30,) | str | file name per recording. The left and right recordings with the same block number were made **at the same time**, so the 30 recordings are only **15 independent blocks** |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**D** = number of columns (16 EMG channels or 20 joints), **k** = number of components.
A good habit: try each function on a tiny made-up array first (e.g. `np.random.rand(5, 16)`).

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:* I think the structure should be confined to some latent states that are reflective othe muscles. I think geometric should reflect better and if there is some way to include some time variance that would be work even better. 

---
## Task 1 · VAF: how much of the EMG is explained?
VAF (*variance accounted for*) is the standard score in the synergy literature:

$$\text{VAF} = 100 \times \left(1 - \frac{\sum (X - \hat X)^2}{\sum X^2}\right)$$

The sums run over **every** entry (all frames and all channels). 100 = perfect rebuild; 0 = no better than
rebuilding everything as 0.

It is **uncentered**: the denominator is `X` itself, not `X` minus its mean. That matters for NMF, which has
no "mean" term: rebuilding the average activity level is part of its job, so it should get credit for it.

In [ ]:
def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Variance accounted for, in %.

    Parameters
    ----------
    X     : (T, D) float, the real data (e.g. EMG, non-negative)
    X_hat : (T, D) float, the rebuilt data (same shape as X)

    Returns
    -------
    float, at most 100. 100 = perfect rebuild. Uncentered: compare the error against X, not X minus its mean.

    Check: vaf(X, X) → 100.0 and vaf(X, np.zeros_like(X)) → 0.0
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. The error at every entry is `X - X_hat`. Square it, then add up **all** of the squares into one number.
2. Do the same for `X` itself: square every entry and add them all up.
3. Divide the first by the second, subtract from 1, multiply by 100.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `a ** 2` squares every entry. `np.sum(a)` with **no** `axis=` adds up every entry into one number.
- Wrap the result in `float(...)` so you return a plain number, not a NumPy scalar.

```python
a = np.array([[1.0, 2.0], [3.0, 0.0]])
np.sum(a ** 2)        # 14.0  (1 + 4 + 9 + 0)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def vaf(X: np.ndarray, X_hat: np.ndarray) -> float:
    err = np.sum((____ - ____) ** 2)
    tot = np.sum(____ ** 2)
    return float(100 * (1 - ____ / ____))
```

</details>

In [ ]:
# ✅ Check Task 1
X_toy = np.random.default_rng(0).random((50, 16))
assert abs(vaf(X_toy, X_toy) - 100) < 1e-9, f"vaf(X, X) = {vaf(X_toy, X_toy)}, expected 100"
assert abs(vaf(X_toy, np.zeros_like(X_toy))) < 1e-9, "vaf(X, zeros) should be 0: did you centre X?"
noisy = X_toy + 0.1
assert abs(vaf(X_toy, noisy) - ref.vaf(X_toy, noisy)) < 1e-9, \
    f"vaf = {vaf(X_toy, noisy):.4f}, expected {ref.vaf(X_toy, noisy):.4f}"
print(f"✅ VAF works. Adding 0.1 to every entry gives VAF = {vaf(X_toy, noisy):.1f}%")

✍️ **Q1.** VAF is *uncentered*. Compute the VAF you get by rebuilding every **test** frame with nothing but the
**training mean** of each channel (`np.tile(X_train.mean(axis=0), (len(X_test), 1))`, after Task 4 defines them,
or on `X_toy` now). What does that number tell you about how to read "90% VAF"?

> *Your answer:*

---
## Task 2 · PCA from scratch, with the SVD
PCA finds the directions along which the data varies most. With the **singular value decomposition (SVD)** it
takes three lines:

1. **Centre:** subtract each column's mean, `Xc = X - mean`. PCA describes variation *around* the average.
2. **SVD:** `U, S, Vt = np.linalg.svd(Xc, full_matrices=False)` splits `Xc = U · diag(S) · Vt`.
   The **rows of `Vt`** are the principal directions (each of length 1, each at right angles to the others),
   already sorted from most to least variance. `S**2` is proportional to the variance along each one.
3. **Keep the top k:** `comps = Vt[:k]`, shape (k, D).

Then two small functions use them:
- **transform** (data → scores): centre, then project onto each direction: `(X - mean) @ comps.T` → (T, k)
- **inverse** (scores → rebuilt data): add up the directions weighted by the scores, then add the mean back:
  `scores @ comps + mean` → (T, D)

**One detail: signs.** An SVD direction and its negative are equally valid, so different computers can flip
them. To make answers comparable, flip each component so its **largest-magnitude entry is positive**.

In [ ]:
def pca_fit(X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    """Fit PCA with k components using the SVD.

    Parameters
    ----------
    X : (T, D) float, data (rows = frames)
    k : int, number of components to keep

    Returns
    -------
    mean  : (D,) float, the mean of each column
    comps : (k, D) float, the top-k principal directions (rows of length 1), most variance first,
            each flipped so that its largest-magnitude entry is positive
    """
    # TODO
    raise NotImplementedError


def pca_transform(X: np.ndarray, mean: np.ndarray, comps: np.ndarray) -> np.ndarray:
    """(T, D) data → (T, k) scores: centre with mean, then project onto comps."""
    # TODO
    raise NotImplementedError


def pca_inverse(scores: np.ndarray, mean: np.ndarray, comps: np.ndarray) -> np.ndarray:
    """(T, k) scores → (T, D) rebuilt data: weighted sum of comps, plus the mean."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `pca_fit`: compute the column means, centre `X`, run the SVD, keep the first `k` rows of `Vt`.
2. Fix the signs: for each kept row, find the entry with the largest absolute value. If that entry is negative,
   multiply the whole row by −1.
3. `pca_transform` and `pca_inverse` are one line each: the two formulas in the text above.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `X.mean(axis=0)` → one mean per column, shape (D,). `X - mean` subtracts it from every row (broadcasting).
- `np.linalg.svd(Xc, full_matrices=False)` returns `U` (T, D), `S` (D,), `Vt` (D, D).
- `np.abs(c).argmax(axis=1)` → for each row, the column of its largest-magnitude entry.
- `np.sign(v)` → +1 or −1. Multiplying `comps * signs[:, None]` flips whole rows.

```python
c = np.array([[0.2, -0.9], [0.6, 0.1]])
big = np.abs(c).argmax(axis=1)          # array([1, 0])
signs = np.sign(c[np.arange(2), big])   # array([-1.,  1.])
c * signs[:, None]                      # [[-0.2, 0.9], [0.6, 0.1]]: row 0 flipped
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def pca_fit(X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    mean = X.mean(axis=____)
    _, _, Vt = np.linalg.svd(X - ____, full_matrices=False)
    comps = Vt[:____]
    big = np.abs(comps).argmax(axis=1)
    signs = np.sign(comps[np.arange(k), ____])
    return mean, comps * signs[:, None]


def pca_transform(X, mean, comps):
    return (X - ____) @ ____.T


def pca_inverse(scores, mean, comps):
    return scores @ ____ + ____
```

</details>

In [ ]:
# ✅ Check Task 2
X_sub = emg[train][:5000]
mean, comps = pca_fit(X_sub, 4)
r_mean, r_comps = ref.pca_fit(X_sub, 4)
assert mean.shape == (16,) and comps.shape == (4, 16), f"shapes {mean.shape}, {comps.shape}; expected (16,), (4, 16)"
assert np.allclose(np.linalg.norm(comps, axis=1), 1), "each component should have length 1"
assert np.allclose(mean, r_mean), "mean differs: did you average down the rows (axis=0)?"
assert np.allclose(comps, r_comps, atol=1e-6), "components differ: check the centring, the [:k] slice and the signs"
scores = pca_transform(X_sub, mean, comps)
assert scores.shape == (5000, 4), f"scores shape {scores.shape}, expected (5000, 4)"
X_back = pca_inverse(scores, mean, comps)
assert np.allclose(X_back, ref.pca_inverse(ref.pca_transform(X_sub, r_mean, r_comps), r_mean, r_comps)), "rebuild differs"
assert np.allclose(pca_inverse(pca_transform(X_sub, *pca_fit(X_sub, 16)), *pca_fit(X_sub, 16)), X_sub), \
    "with all 16 components the rebuild should be perfect"
print(f"✅ PCA works. 4 components explain VAF = {vaf(X_sub, X_back):.1f}% of these frames")

### 📊 What the PCA components look like
Each bar chart is one component: a weight per electrode. Notice the **negative** weights. PCA describes
variation as "more of this electrode *and less of that one*", but a muscle can't be *negatively* active.
That is the reason for NMF in the next task.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 2.8), sharey=True)
for i in range(4):
    ax[i].bar(range(16), comps[i], color=np.where(comps[i] >= 0, "C0", "C3"))
    ax[i].axhline(0, color="k", lw=0.5); ax[i].set_title(f"PC {i + 1}"); ax[i].set_xlabel("electrode")
ax[0].set_ylabel("weight")
plt.tight_layout(); plt.show()

✍️ **Q2.** PCA components have negative weights. What would a negative weight on an electrode mean
*physically*, if a component were a muscle synergy? Why is that a problem for calling PCA components "synergies"?

> *Your answer:*

---
## Task 3 · NMF from scratch: Lee & Seung's multiplicative updates
NMF looks for `X ≈ C W` where **everything is non-negative**: `W` (k × 16) are the synergies (how much each
electrode takes part) and `C` (T × k) the activations (how active each synergy is at each frame). Muscle
activity can only add up, never cancel, so this matches the biology better than PCA.

**The goal:** make the squared error `‖X − C W‖²` (the sum of all squared differences) as small as possible.

**The algorithm** (Lee & Seung 2001). Start from random positive `C` and `W`, then repeat:

$$W \leftarrow W \odot \frac{C^\top X}{C^\top C\, W} \qquad\qquad C \leftarrow C \odot \frac{X W^\top}{C\, W W^\top}$$

`⊙` and the fraction are **element by element** (`*` and `/` in NumPy); the products inside are matrix
products (`@`).

**Why it works.** Look at one entry of `W`. The top of the fraction (`CᵀX`) measures how much the data *asks*
for that entry; the bottom (`CᵀCW`) how much the current model already *gives*. If the data asks for more,
the ratio is above 1 and the entry grows; if the model gives too much, it shrinks. When they agree the
ratio is 1 and nothing changes. Lee & Seung proved each update can only lower the error. And because you
only ever multiply positive numbers, everything stays non-negative automatically: no clipping needed.

**Details that make it work in practice**
- **Start:** `rng = np.random.default_rng(seed)`, `scale = np.sqrt(X.mean() / k)`,
  then `C = scale * rng.random((T, k))` and `W = scale * rng.random((k, D))`, in that order. The scale makes
  `C @ W` start out about the size of `X`. (Use exactly this recipe and your answer will match the reference.)
- **Never divide by 0:** add a tiny number to every denominator, e.g. `+ 1e-10`.
- **Track the error:** after each iteration append `np.sum((X - C @ W) ** 2)` to a list, so you can see it fall.
- **Tidy up the scale:** `C W` doesn't change if you make a row of `W` 10× bigger and the matching column of `C`
  10× smaller. So at the end, scale every row of `W` to length 1 and multiply `C`'s columns by the old lengths.
  Then synergies from different fits are comparable.

**New data with fixed synergies** (`nmf_transform`): for the test set you keep `W` and only find `C`. That is
the same `C` update alone, repeated, starting from a random `C`.

In [ ]:
EPS = 1e-10   # added to every denominator so nothing divides by 0


def nmf_fit(X: np.ndarray, k: int, n_iter: int = 300, seed: int = 0
            ) -> tuple[np.ndarray, np.ndarray, list[float]]:
    """Non-negative matrix factorisation X ≈ C @ W with multiplicative updates.

    Parameters
    ----------
    X      : (T, D) float, non-negative data
    k      : int, number of synergies
    n_iter : int, number of update rounds (each round: update W, then update C)
    seed   : int, for the random start

    Returns
    -------
    C      : (T, k) float ≥ 0, activations
    W      : (k, D) float ≥ 0, synergies, each row scaled to length 1 (C scaled up to compensate)
    losses : list of n_iter floats, the squared error sum((X - C @ W) ** 2) after each round
    """
    # TODO
    raise NotImplementedError


def nmf_transform(X: np.ndarray, W: np.ndarray, n_iter: int = 300, seed: int = 0) -> np.ndarray:
    """Activations for new data with the synergies held fixed.

    Parameters
    ----------
    X : (T, D) float, non-negative data
    W : (k, D) float, synergies from nmf_fit (NOT changed here)

    Returns
    -------
    C : (T, k) float ≥ 0, so that C @ W ≈ X
    Start C the same way as nmf_fit (same seed recipe), then repeat only the C update.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Start: make the random generator, the scale, then `C` and `W` (the recipe in the text above).
2. Loop `n_iter` times: update `W` with its formula, then update `C` with its formula (using the new `W`),
   then append the current squared error to `losses`.
3. After the loop: compute the length of each row of `W`, divide the rows by it, multiply `C`'s columns by it.
4. `nmf_transform`: the same start for `C` (the `W` you draw is thrown away; keep the one passed in),
   then loop the `C` update only.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `A @ B` is a matrix product; `A * B` and `A / B` work entry by entry.
- `W *= top / (bottom + EPS)` updates `W` in place.
- `np.linalg.norm(W, axis=1)` → the length of each row, shape (k,). `W / n[:, None]` divides row i by `n[i]`;
  `C * n` multiplies column i of `C` by `n[i]`.

```python
C = np.array([[1.0, 2.0]]); W = np.array([[3.0, 0.0], [0.0, 4.0]])
n = np.linalg.norm(W, axis=1)   # array([3., 4.])
(C * n) @ (W / n[:, None])      # [[3., 8.]]: the same as C @ W
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def nmf_fit(X, k, n_iter=300, seed=0):
    rng = np.random.default_rng(seed)
    scale = np.sqrt(X.mean() / k)
    C = scale * rng.random((X.shape[0], k))
    W = scale * rng.random((k, X.shape[1]))
    losses = []
    for _ in range(n_iter):
        W *= (C.T @ ____) / (C.T @ C @ ____ + EPS)
        C *= (X @ ____.T) / (C @ W @ ____.T + EPS)
        losses.append(float(np.sum((X - ____ @ ____) ** 2)))
    n = np.linalg.norm(W, axis=____) + EPS
    return C * n, W / n[:, None], losses


def nmf_transform(X, W, n_iter=300, seed=0):
    rng = np.random.default_rng(seed)
    scale = np.sqrt(X.mean() / W.shape[0])
    C = scale * rng.random((X.shape[0], W.shape[0]))
    for _ in range(n_iter):
        C *= ____ / (____ + EPS)
    return C
```

</details>

In [ ]:
# ✅ Check Task 3 (about 10 s)
X_sub = emg[train][:20000]
C, W, losses = nmf_fit(X_sub, 4, n_iter=200, seed=SEED)
assert C.shape == (20000, 4) and W.shape == (4, 16), f"shapes {C.shape}, {W.shape}; expected (20000, 4), (4, 16)"
assert (C >= 0).all() and (W >= 0).all(), "C and W must stay non-negative"
assert len(losses) == 200, f"{len(losses)} losses, expected one per iteration (200)"
assert all(b <= a * (1 + 1e-9) for a, b in zip(losses, losses[1:])), \
    "the error went UP at some step: check the order of the updates and the formulas"
assert np.allclose(np.linalg.norm(W, axis=1), 1, atol=1e-6), "each row of W should have length 1 at the end"
r_C, r_W, r_losses = ref.nmf_fit(X_sub, 4, n_iter=200, seed=SEED)
assert abs(vaf(X_sub, C @ W) - vaf(X_sub, r_C @ r_W)) < 0.5, \
    f"VAF {vaf(X_sub, C @ W):.2f}% vs reference {vaf(X_sub, r_C @ r_W):.2f}%: the updates may be wrong"
C_new = nmf_transform(X_sub[:2000], W, n_iter=200, seed=SEED)
assert C_new.shape == (2000, 4) and (C_new >= 0).all()
assert abs(vaf(X_sub[:2000], C_new @ W) - vaf(X_sub[:2000], C[:2000] @ W)) < 1.0, \
    "nmf_transform should rebuild about as well as the fit itself"
same = np.allclose(W, r_W, atol=1e-6)
print(f"✅ NMF works. VAF with 4 synergies = {vaf(X_sub, C @ W):.1f}%"
      + (" (identical to the reference)" if same else " (close to the reference; a different start is fine)"))

### 📊 The error falls, and the synergies are all positive
Left: the squared error after each round. It drops fast, then flattens: more rounds would barely help.
Right: the 4 synergies. Compare with the PCA components above: no negative weights, and each synergy
tends to sit on a **group of neighbouring electrodes** (one region of the forearm).

In [ ]:
fig = plt.figure(figsize=(14, 3))
ax0 = fig.add_subplot(1, 5, 1)
ax0.plot(losses); ax0.set_yscale("log"); ax0.set_xlabel("iteration"); ax0.set_title("squared error")
for i in range(4):
    a = fig.add_subplot(1, 5, i + 2)
    a.bar(range(16), W[i], color="C2"); a.set_title(f"synergy {i + 1}"); a.set_xlabel("electrode")
plt.tight_layout(); plt.show()

✍️ **Q3.** Look at the update rules. (a) Why can an entry of `W` never become negative? (b) If an entry
becomes exactly 0, what happens to it in every later round? Why is a *random* start therefore better than
starting from zeros? (c) Why scale the rows of `W` to length 1 at the end?

> *Your answer:*

---
## Task 4 · NMF and PCA for k = 1 … 10, scored on test data
Now use your own functions on the full data. For each k:
- **NMF:** `nmf_fit` on `X_train`, then `nmf_transform` on `X_test` with that `W`, then VAF on `X_test`.
- **PCA:** `pca_fit` on `X_train`, then `pca_inverse(pca_transform(X_test, …), …)`, then VAF on `X_test`.

Fitting on train and scoring on test asks whether the synergies **generalise** to later data, not just whether
they can memorise the training frames. This takes about a minute (NMF is the slow part).

In [ ]:
ks = list(range(1, 11))
vaf_nmf: list[float] = []      # length 10 when done: test VAF (%) for k = 1 … 10
vaf_pca: list[float] = []
nmf_W: dict[int, np.ndarray] = {}                       # k → W (k, 16)            (Task 8 reuses it)
pca_fits: dict[int, tuple[np.ndarray, np.ndarray]] = {} # k → (mean, comps)

X_train = emg[train]           # (T_train, 16)
X_test  = emg[test]            # (T_test, 16)

for k in ks:
    # TODO: NMF: fit on X_train (seed=SEED), store W in nmf_W[k], rebuild X_test with nmf_transform, append the VAF
    # TODO: PCA: fit on X_train, store (mean, comps) in pca_fits[k], rebuild X_test, append the VAF
    ...
    print(f"k = {k:2d}   NMF {vaf_nmf[-1]:5.1f}%   PCA {vaf_pca[-1]:5.1f}%")

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

Inside the loop, two blocks of three lines each:
1. **NMF:** fit → keep `W` → find test activations with `W` fixed → rebuild as `C_test @ W` → VAF against `X_test`.
2. **PCA:** fit → keep `(mean, comps)` → scores of `X_test` → rebuild → VAF against `X_test`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `_` is the usual name for a returned value you don't need: `_, W, _ = nmf_fit(...)`.
- `pca_fit` returns a tuple, so you can unpack it straight back in: `pca_transform(X, *pca_fits[k])`.

```python
fit = (np.zeros(2), np.eye(2))     # (mean, comps)
mean, comps = fit                  # unpack
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for k in ks:
    _, W, _ = nmf_fit(____, k, seed=SEED)
    nmf_W[k] = W
    C_test = nmf_transform(____, W, seed=SEED)
    vaf_nmf.append(vaf(X_test, ____ @ ____))

    mean, comps = pca_fit(____, k)
    pca_fits[k] = (mean, comps)
    X_hat = pca_inverse(pca_transform(____, mean, comps), mean, comps)
    vaf_pca.append(vaf(____, ____))
    print(f"k = {k:2d}   NMF {vaf_nmf[-1]:5.1f}%   PCA {vaf_pca[-1]:5.1f}%")
```

</details>

In [ ]:
# ✅ Check Task 4 (about 10 s)
assert len(vaf_nmf) == len(vaf_pca) == 10, "one VAF per k = 1 … 10 for each method"
r_pca = [vaf(X_test, ref.pca_inverse(ref.pca_transform(X_test, *ref.pca_fit(X_train, k)), *ref.pca_fit(X_train, k)))
         for k in ks]
assert np.allclose(vaf_pca, r_pca, atol=1e-6), "PCA VAFs differ from the reference: fit on X_train, score on X_test"
_, r_W4, _ = ref.nmf_fit(X_train, 4, seed=SEED)
r_nmf4 = vaf(X_test, ref.nmf_transform(X_test, r_W4, seed=SEED) @ r_W4)
assert abs(vaf_nmf[3] - r_nmf4) < 0.5, f"NMF VAF at k = 4 is {vaf_nmf[3]:.2f}%, reference {r_nmf4:.2f}%"
print("✅ VAF curves match the reference")

✍️ **Q4.** (a) The PCA curve is at or above the NMF curve at every k. Why must that be true (think about what
PCA is optimal for, and what NMF is not allowed to do)? (b) On the test set we **refit `C`** freely and only
keep `W` fixed. Is that a strict test of the synergies? Try it: make a *random* non-negative `W` (4 × 16, rows of
length 1) and score it on `X_test` with `nmf_transform`. How far above it is the real NMF?

> *Your answer:*

---
## Task 5 · Fairer reference points: centred VAF and random synergies
Two things make the Task 4 numbers look better than they are (Q1 and Q4 hinted at both):
1. **Uncentered VAF gives credit for the mean.** **Centred VAF** compares the error with the data *minus its
   column means*, so only the moment-to-moment variation counts:
   $$\text{VAF}_c = 100 \times \left(1 - \frac{\sum (X - \hat X)^2}{\sum (X - \bar X)^2}\right)$$
2. **`C` is refit on the test frames.** Any non-negative `W` gets some credit for free. The honest reference is
   a **null**: random non-negative synergies of the same size, with `C` refit in exactly the same way. Real
   synergies only mean something by **how far they beat this null**.

This task adds three curves to Task 4's: centred VAF for NMF and PCA, and (uncentred) VAF for random `W`.
It reuses `nmf_W` and `pca_fits`, so nothing is refit (about 30 s).

In [ ]:
def vaf_centred(X: np.ndarray, X_hat: np.ndarray) -> float:
    """Centred variance accounted for, in %.

    Parameters
    ----------
    X, X_hat : (T, D) float, real and rebuilt data

    Returns
    -------
    float, at most 100. Like vaf, but the denominator is the sum of squares of X minus its column means.
    Check: vaf_centred(X, np.tile(X.mean(axis=0), (len(X), 1))) → 0.0
    """
    # TODO
    raise NotImplementedError


def random_W(k: int, D: int, seed: int) -> np.ndarray:
    """k random non-negative synergies (k, D), each row scaled to length 1.

    Use rng = np.random.default_rng(seed) and rng.random((k, D)), so your answer matches the reference.
    """
    # TODO
    raise NotImplementedError


vaf_nmf_c: list[float] = []    # length 10: centred test VAF of NMF for k = 1 … 10
vaf_pca_c: list[float] = []    # length 10: centred test VAF of PCA
vaf_rand: list[float] = []     # length 10: (uncentred) test VAF of random_W(k, 16, SEED), C refit with nmf_transform
vaf_mean_only = None           # float: VAF of rebuilding every test frame with the training mean (Q1)
for k in ks:
    # TODO: NMF rebuild of X_test with nmf_W[k] → centred VAF; PCA rebuild with pca_fits[k] → centred VAF;
    # TODO: random W → nmf_transform → VAF
    ...

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `vaf_centred` is your `vaf` with one change: subtract the column means from `X` in the denominator.
2. `random_W`: draw a (k, D) array of uniform numbers in [0, 1), then divide each row by its length.
3. In the loop, rebuild `X_test` three ways (NMF with `nmf_W[k]`, PCA with `pca_fits[k]`, random `W` with
   `nmf_transform`) and append the scores.
4. `vaf_mean_only`: `np.tile(X_train.mean(axis=0), (len(X_test), 1))` is the "rebuild".

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `X.mean(axis=0)` → (D,) column means; `X - X.mean(axis=0)` centres every column.
- `W / np.linalg.norm(W, axis=1, keepdims=True)` scales each row to length 1.

```python
X = np.array([[1.0, 2.0], [3.0, 4.0]])
X - X.mean(axis=0)          # [[-1., -1.], [ 1.,  1.]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def vaf_centred(X, X_hat):
    return float(100 * (1 - np.sum((X - X_hat) ** 2) / np.sum((X - X.mean(axis=____)) ** 2)))

def random_W(k, D, seed):
    W = np.random.default_rng(seed).random((____, ____))
    return W / np.linalg.norm(W, axis=1, keepdims=____)

for k in ks:
    W = nmf_W[k]
    vaf_nmf_c.append(vaf_centred(X_test, nmf_transform(X_test, W, seed=SEED) @ ____))
    vaf_pca_c.append(vaf_centred(X_test, pca_inverse(pca_transform(X_test, *pca_fits[k]), *pca_fits[____])))
    Wr = random_W(k, 16, SEED)
    vaf_rand.append(vaf(X_test, nmf_transform(X_test, ____, seed=SEED) @ Wr))
vaf_mean_only = vaf(X_test, np.tile(X_train.mean(axis=0), (len(X_test), 1)))
```

</details>

In [ ]:
# ✅ Check Task 5
X_toy = np.random.default_rng(0).random((50, 16))
assert abs(vaf_centred(X_toy, np.tile(X_toy.mean(axis=0), (50, 1)))) < 1e-9, "the mean alone should score 0 centred VAF"
assert abs(vaf_centred(X_toy, X_toy + 0.05) - ref.vaf_centred(X_toy, X_toy + 0.05)) < 1e-9
assert np.allclose(random_W(4, 16, SEED), ref.random_W(4, 16, SEED)), "use np.random.default_rng(seed).random((k, D))"
assert len(vaf_nmf_c) == len(vaf_pca_c) == len(vaf_rand) == 10
assert abs(vaf_pca_c[3] - ref.vaf_centred(X_test, ref.pca_inverse(ref.pca_transform(X_test, *pca_fits[4]), *pca_fits[4]))) < 1e-6
Wr4 = ref.random_W(4, 16, SEED)
assert abs(vaf_rand[3] - ref.vaf(X_test, ref.nmf_transform(X_test, Wr4, seed=SEED) @ Wr4)) < 0.5
print(f"✅ At k = 4: NMF {vaf_nmf[3]:.1f}% uncentred / {vaf_nmf_c[3]:.1f}% centred; random W {vaf_rand[3]:.1f}%; "
      f"mean only {vaf_mean_only:.1f}%")

✍️ **Q5.** At k = 4, how many points does NMF beat random synergies by, and how many points does it beat the mean-only
rebuild by? Which of the three numbers (VAF, centred VAF, gap over random) would you put in a paper, and why?

> *Your answer:*

---
## Task 6 · Plot VAF against k and choose k
Two panels. **Left:** uncentred VAF (%) against k for NMF, PCA and **random synergies** (the null, dashed),
with a dotted line at the 90% threshold. **Right:** centred VAF for NMF and PCA. Choose k where the NMF curve
**flattens** (adding a synergy barely helps any more), and look at where it first passes 90%. Say which rule you
used: they may disagree, and that's worth discussing with the team. Task 7 then checks your choice on held-out stages.

In [ ]:
# TODO: left panel: vaf_nmf, vaf_pca and vaf_rand (dashed) against ks, with a dotted line at 90%
# TODO: right panel: vaf_nmf_c and vaf_pca_c against ks
# TODO: choose k; write it here and in the Result cell at the bottom

K_CHOSEN: int | None = None    # e.g. 4

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Make one axis, draw two lines (NMF, PCA) against `ks`, add a horizontal dashed line at 90.
2. Label the axes, add a legend.
3. Read where the NMF curve flattens and where it crosses 90%. A helpful extra: plot the **gain** from each extra
   synergy, `np.diff(vaf_nmf)`, and see where it drops below about 1–2 points.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `ax.plot(x, y, marker="o", label="NMF")` draws a line with dots. `ax.axhline(90, ls="--", color="gray")`.
- `ax.legend()` shows the labels.

```python
np.diff([77, 85, 89, 92, 93])     # array([8, 4, 3, 1]): the gain from each extra component
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, ____, marker="o", label="NMF")
ax[0].plot(ks, ____, marker="s", label="PCA")
ax[0].plot(ks, ____, ls="--", color="gray", label="random W (null)")
ax[0].axhline(____, ls=":", color="gray")
ax[0].set_xlabel("k"); ax[0].set_ylabel("test VAF (%)"); ax[0].legend()
ax[1].plot(ks, vaf_nmf_c, marker="o", label="NMF"); ax[1].plot(ks, vaf_pca_c, marker="s", label="PCA")
ax[1].set_xlabel("k"); ax[1].set_ylabel("centred test VAF (%)"); ax[1].legend(); plt.show()
K_CHOSEN = ____
```

</details>

✍️ **Q6.** Which rule did you use to choose k (flattening, 90%, or something else), and what result would make
you change it? Would two people looking at this curve pick the same k?

> *Your answer:*

---
## Task 7 · Choose k on held-out stages (leave-one-stage-out)
The curves above are fit on the start of each recording and scored on its end: the test frames come from the
**same tasks**. A stricter test of "how many synergies does this person use" is to hold out a **whole stage**:
fit on all the other stages, score on the held-out one, and repeat for every stage.

**Independent units.** The 30 recordings are only **15 independent blocks**: the left and right recordings of a
block were made **at the same time** (same block number in `filename_of_rec`). Holding out a stage must also hold
out anything recorded simultaneously with it, so the shared protocol (`protocol.leave_one_stage_out`, with
`filename=`) merges `OneHandedFreeStyle` with the idle-hand `unconstrained` recording made at the same time. That
gives **14 folds**. Every fold gives one number per k, and the spread over folds gives a **95% CI**
(`protocol.session_bootstrap_ci`, which resamples folds, not frames).

**Rule for k:** the smallest k after which one more synergy adds less than **1 VAF point** on held-out stages
(mean over folds). Report the gap over random synergies at that k, with its CI.

**Keeping it fast:** 14 folds × 7 values of k = 98 NMF fits. Using every **10th** frame and **100** update rounds
keeps this to about a minute; neighbouring frames are near-copies (20 ms apart, smoothed at 10 Hz), so little is lost.

In [ ]:
def heldout_vaf_table(X: np.ndarray, folds: list[tuple[np.ndarray, np.ndarray]], ks: list[int],
                      step: int = 10, n_iter: int = 100, seed: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Held-out VAF for every fold and k, for NMF and for random_W of the same size.

    Parameters
    ----------
    X     : (N, 16) EMG
    folds : list of (train_mask, test_mask), each (N,) bool
    ks    : values of k to try
    step, n_iter : use X[mask][::step] and n_iter NMF rounds (both for nmf_fit and nmf_transform)

    Returns
    -------
    real, rand : each (n_folds, len(ks)) float, VAF (%) on the test frames of each fold
    """
    # TODO
    raise NotImplementedError


ks_loso = list(range(1, 8))
folds = [(tr, te) for _, tr, te in protocol.leave_one_stage_out(stage_of_rec, rec, filename=filename_of_rec)]
real_loso, rand_loso = None, None      # TODO: heldout_vaf_table(emg, folds, ks_loso, seed=SEED)
K_LOSO: int | None = None              # TODO: smallest k whose next synergy adds < 1 point (mean over folds)
gap_ci = None                          # TODO: protocol.session_bootstrap_ci(real − rand at K_LOSO, per fold)

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Loop over folds and over k. In each fold, take every `step`-th training and test frame.
2. Fit NMF on the training frames, then rebuild the test frames with `nmf_transform` and score with `vaf`.
3. Do the same rebuild with `random_W(k, 16, seed)` instead of the fitted `W`.
4. `K_LOSO`: `np.diff(real_loso.mean(axis=0))` gives the gain of each extra synergy; find the first gain below 1.
5. `gap_ci`: the per-fold difference `real_loso[:, j] - rand_loso[:, j]` at that k goes into the bootstrap.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.zeros((len(folds), len(ks)))` makes the empty tables; fill `real[f, j]`.
- `for f, (tr, te) in enumerate(folds):` gives the fold number and its two masks.
- `protocol.session_bootstrap_ci(values)` → `(mean, low, high)`.

```python
gains = np.diff([75, 80, 87, 90, 92.5, 93.8, 94.0])   # [5, 7, 3, 2.5, 1.3, 0.2]
next(i for i, g in enumerate(gains) if g < 1)          # 5 → k = ks[5] = 6
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def heldout_vaf_table(X, folds, ks, step=10, n_iter=100, seed=0):
    real = np.zeros((len(folds), len(ks))); rand = np.zeros_like(real)
    for f, (tr, te) in enumerate(folds):
        X_tr, X_te = X[tr][::step], X[te][::step]
        for j, k in enumerate(ks):
            _, W, _ = nmf_fit(____, k, n_iter=n_iter, seed=seed)
            real[f, j] = vaf(X_te, nmf_transform(____, W, n_iter=n_iter, seed=seed) @ W)
            Wr = random_W(k, X.shape[1], seed)
            rand[f, j] = vaf(X_te, nmf_transform(X_te, ____, n_iter=n_iter, seed=seed) @ Wr)
    return real, rand

real_loso, rand_loso = heldout_vaf_table(emg, folds, ks_loso, seed=SEED)
gains = np.diff(real_loso.mean(axis=0))
K_LOSO = ks_loso[next((i for i, g in enumerate(gains) if g < ____), len(gains))]
j = ks_loso.index(K_LOSO)
gap_ci = protocol.session_bootstrap_ci(real_loso[:, j] - rand_loso[:, ____])
```

</details>

In [ ]:
# ✅ Check Task 7 (about 10 s)
assert len(folds) == 14, "14 folds: OneHandedFreeStyle and unconstrained were recorded together and are held out together"
assert real_loso.shape == rand_loso.shape == (14, len(ks_loso))
r_real, r_rand = ref.heldout_vaf_table(emg, folds[:1], [4], step=10, n_iter=100, seed=SEED)
assert abs(real_loso[0, 3] - r_real[0, 0]) < 0.5 and abs(rand_loso[0, 3] - r_rand[0, 0]) < 0.5, \
    f"fold 0, k = 4: yours {real_loso[0, 3]:.2f} / {rand_loso[0, 3]:.2f}, reference {r_real[0, 0]:.2f} / {r_rand[0, 0]:.2f}"
assert K_LOSO == ref.choose_k(real_loso, ks_loso), f"K_LOSO = {K_LOSO}, expected {ref.choose_k(real_loso, ks_loso)}"
m, lo, hi = gap_ci
print(f"✅ Leave-one-stage-out chooses k = {K_LOSO}; NMF beats random synergies there by {m:.1f} points "
      f"(95% CI {lo:.1f} to {hi:.1f}, over 14 held-out stage groups)")

### 📊 Held-out stages: real vs random synergies, and the gain from each extra synergy
Left: mean held-out VAF ± spread over folds, for NMF and for random synergies. Right: how much each extra synergy
adds, with the 1-point rule. **What to look for:** a wide, steady gap between the two curves (the synergies are
real structure), and where the right-hand bars drop below the line (your k).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for tab, lab, c in [(real_loso, "NMF", "C0"), (rand_loso, "random W (null)", "gray")]:
    ax[0].errorbar(ks_loso, tab.mean(axis=0), yerr=tab.std(axis=0), marker="o", capsize=3, label=lab, color=c)
ax[0].set_xlabel("k"); ax[0].set_ylabel("held-out-stage VAF (%)"); ax[0].legend()
g = np.diff(real_loso, axis=1)
ax[1].bar(ks_loso[1:], g.mean(axis=0), yerr=g.std(axis=0), capsize=3)
ax[1].axhline(1, ls="--", color="gray"); ax[1].set_xlabel("k (gain from k − 1 to k)"); ax[1].set_ylabel("VAF points")
plt.tight_layout(); plt.show()

✍️ **Q7.** Does leave-one-stage-out pick the same k as your Task 6 choice? If not, which would you trust for the decoder, and
why? Why does the protocol hold out `OneHandedFreeStyle` and `unconstrained` together, and what would go wrong if
it didn't?

> *Your answer:*

---
## Task 8 · Look at the synergies
For the chosen k:
1. A bar chart per synergy of its 16 electrode weights (`nmf_W[K_CHOSEN]`, rows already length 1).
2. For one recording: the activations `C` over time, with 3 joint angles underneath on the same time axis.
   When does each synergy switch on? Does it line up with a particular finger moving?

In [ ]:
W = nmf_W[K_CHOSEN]                      # (k, 16): row i = synergy i
r = 0                                    # the recording to plot
in_r = rec == r                          # (N,) bool mask for that recording
t_r = np.arange(in_r.sum()) / 50         # time axis in seconds (50 frames/s)

# TODO: C_r = nmf_transform(emg[in_r], W, seed=SEED)    → (T_r, k) activations of recording r
# TODO: a row of bar charts, one per synergy (its 16 weights)
# TODO: a 2-row figure sharing the x axis: C_r over time (top); pose[in_r] for 3 joints of your choice (bottom)

#### 💡 Stuck on Task 8? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Compute the activations of the recording with your `nmf_transform` and the chosen `W`.
2. Figure 1: `plt.subplots(1, K_CHOSEN)` and a `bar` per synergy.
3. Figure 2: `plt.subplots(2, 1, sharex=True)`: one line per synergy on top, three joint angles underneath.
   Masked pose frames (`pose_valid` False) sit at 0°; you can blank them with `np.where(pose_valid[in_r], …, np.nan)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `plt.subplots(2, 1, sharex=True)` → two axes stacked; zooming one zooms both.
- `ax.plot(t, Y)` with a 2-D `Y` (T, n) draws one line per column.

```python
y = np.array([1.0, 2.0, 3.0]); ok = np.array([True, False, True])
np.where(ok, y, np.nan)    # array([ 1., nan,  3.]): NaN points are not drawn
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
C_r = nmf_transform(emg[in_r], ____, seed=SEED)
fig, ax = plt.subplots(1, K_CHOSEN, figsize=(14, 2.5), sharey=True)
for i in range(K_CHOSEN):
    ax[i].bar(range(16), ____[i]); ax[i].set_title(f"synergy {i + 1}")
plt.show()

joints = [5, 9, 13]                         # INDEX_, MIDDLE_, RING_MCP_FE
P_r = np.where(pose_valid[in_r][:, None], pose[in_r][:, joints], np.nan)
fig, ax = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
ax[0].plot(t_r, ____); ax[0].set_ylabel("activation")
ax[1].plot(t_r, ____); ax[1].set_ylabel("angle (°)"); ax[1].set_xlabel("time (s)")
plt.show()
```

</details>

✍️ **Q8.** Does any synergy switch on with a particular finger movement? Does a synergy sit on neighbouring
electrodes (one region of the forearm)? What else, apart from a shared neural command, could make
neighbouring electrodes rise together?

> *Your answer:*

---
## Task 9 · Eigenpostures: how many dimensions does the pose need?
The proposal's **pose bottleneck**: the decoder predicts a few PCA scores, and PCA turns them back into 20 angles.
Fingers share tendons, so joint angles move together and a few components should rebuild them well.

Your `pca_fit` works on any (T, D) data, so use it again on the pose: fit on **valid** training poses only
(`pose_valid`), rebuild the **test** poses with k = 1 … 20 components, and measure the error in degrees.

**Choosing `K_POSE` is a judgement call; agree it as a team.** The bottleneck's error adds to the decoder's,
so compare it with the error we want to beat (the best published model is off by 15.63°) and with a decoder
that always guesses the average pose. The proposal's starting rule was **under 2°**: check whether that is
reachable with *few* components, and if not, say what that means for the eigenposture idea (Santello et al.
reported that 2 components explain most of the *variance* of static grasps: is mean error in degrees on
free movement the same question?).

In [ ]:
P_train = pose[train & pose_valid]   # (T_train_valid, 20) degrees
P_test  = pose[test & pose_valid]    # (T_test_valid, 20)


def pose_error_deg(Y: np.ndarray, Y_hat: np.ndarray) -> float:
    """Mean absolute joint-angle error in degrees.

    Parameters
    ----------
    Y     : (T, 20) float, true joint angles in degrees
    Y_hat : (T, 20) float, rebuilt joint angles in degrees

    Returns
    -------
    float, averaged over all joints and frames. Check: pose_error_deg(Y, Y) → 0.0
    """
    # TODO
    raise NotImplementedError


ks_pose = list(range(1, 21))
err_pose: list[float] = []                              # length 20 when done: test error (°) for k = 1 … 20
pose_fits: dict[int, tuple[np.ndarray, np.ndarray]] = {}  # k → (mean, comps)

for k in ks_pose:
    # TODO: pca_fit on P_train, store it in pose_fits[k]
    # TODO: rebuild P_test, append pose_error_deg(P_test, rebuilt) to err_pose
    ...

# TODO: plot err_pose against ks_pose, with a dashed line at your cut-off
K_POSE: int | None = None            # TODO: your choice, and the reason in the Result cell

#### 💡 Stuck on Task 9? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `pose_error_deg`: the absolute difference at every entry, then the mean of all of them.
2. The loop is the PCA half of Task 4, on `P_train` / `P_test`, with `pose_error_deg` instead of `vaf`.
3. Plot like Task 6, with the dashed line at your cut-off, and pick the first k below it.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.abs(a)` → absolute value of every entry; `np.mean(a)` with no `axis=` → one number.
- To find the first k under a cut-off: `next(k for k, e in zip(ks_pose, err_pose) if e < 2.0)`.

```python
np.mean(np.abs(np.array([[1.0, -3.0]]) - 0))   # 2.0
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def pose_error_deg(Y, Y_hat):
    return float(np.mean(np.abs(____ - ____)))

for k in ks_pose:
    mean, comps = pca_fit(____, k)
    pose_fits[k] = (mean, comps)
    P_hat = pca_inverse(pca_transform(____, mean, comps), mean, comps)
    err_pose.append(pose_error_deg(____, ____))
```

</details>

In [ ]:
# ✅ Check Task 9
Y_toy = np.random.default_rng(1).normal(size=(30, 20))
assert pose_error_deg(Y_toy, Y_toy) == 0.0
assert abs(pose_error_deg(Y_toy, Y_toy + 2) - 2.0) < 1e-9, "an error of 2° everywhere should give 2.0"
assert len(err_pose) == 20, "one error per k = 1 … 20"
r_err = [ref.pose_error_deg(P_test, ref.pca_inverse(ref.pca_transform(P_test, *ref.pca_fit(P_train, k)),
                                                    *ref.pca_fit(P_train, k))) for k in ks_pose]
assert np.allclose(err_pose, r_err, atol=1e-6), "errors differ from the reference: fit on P_train, score on P_test"
print("✅ Pose errors (°): " + ", ".join(f"k={k}: {e:.1f}" for k, e in zip(ks_pose, err_pose)))

✍️ **Q9.** The pose needs many more PCA components than the EMG needs synergies to reach a small error. Give
two possible reasons (think about what 16 electrodes on the skin can resolve, how independent the fingers are,
and what is in the tracking data). Santello et al. found 2 components explain most of the variance of *static
grasps*. Is mean error in degrees on *free movement* the same question?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **15 independent blocks, not 30 recordings** | The left- and right-hand recordings of a block were made at the same time, and blocks 14–15 pair OneHandedFreeStyle with the idle hand's 'unconstrained' recording. Treating 30 recordings (or frames) as independent overstates certainty | Every CI here resamples held-out stage groups (14 folds) via `protocol`, with `filename=` |
| **One person** | All 15 blocks are the same participant | Nothing here says how many synergies *other* people need; that needs the full dataset (stage 7) |
| **Uncentred VAF is generous** | The training mean alone scores well above 0 (Task 5) | Report centred VAF and the gap over random synergies, not only VAF |
| **`C` is refit on the test frames** | Random synergies already score high, and VAF can only rise with k | The random-W null (Task 5, Task 7); for an over-fitting-aware k, **bi-cross-validation** (hold out channels *and* frames, Owen & Perry 2009) |
| **Time split vs held-out stages** | The time split tests later frames of the same tasks | Task 7 holds out whole stage groups; prefer its k |
| **Left and right hands pooled** | The band is mirrored on the other wrist, so electrode *j* sits over a different muscle on each hand | Fit each hand separately (notebook 03) and compare k |
| **Speed shortcuts in Task 7** | Every 10th frame and 100 NMF rounds make each fold's number slightly noisier | Rerun a borderline k at full size |
| **Noise floor** | Rectified background noise is a positive level that NMF may spend a synergy on | Subtract each channel's resting level (e.g. 5th percentile) and refit |
| **Loud channels dominate VAF** | One scale per recording, so loud electrodes carry more squared error | Repeat with per-channel scaling; does k change? |
| **Pose error includes tracking noise** | Small eigenpostures may fit camera/IK jitter | Low-pass the pose at ~10 Hz before PCA, or report variance explained as well as degrees |

---
## Result · gate for stage 1
**Question:** Is the EMG low-dimensional beyond chance, and how many synergies (k) and eigenpostures (K_POSE)?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| k chosen on the time split (Task 6) | | | |
| k chosen by leave-one-stage-out (Task 7) | | | |
| VAF / centred VAF at that k (NMF; PCA) | | | |
| Gap over random synergies at that k (Task 7 CI) | | | |
| Chosen K_POSE and pose error at K_POSE (°) | | | |

**Gate:** the gap over random synergies at the leave-one-stage-out k has a 95% CI above 0 (and is large, e.g. > 5 points).

- **Pass →** stage 2 (notebook 03): are the synergies stable and not crosstalk? Use the leave-one-stage-out k.
- **Fail →** the EMG is not lower-dimensional than chance co-activation: stop calling the components synergies; stages 4–5 can still ask what the EMG encodes.
- **Was your prediction right? Why or why not?**